# 올랐는데 아직 비관적? RBSI를 두 축으로 읽기

대한상공회의소 2026년 4분기 공식 집계표로 **차트 4개**를 재현합니다. 응답자 원시자료가 아니라 공식 표·그림을 전사한 작은 데이터입니다. 전망지수이며 실제 매출이 아닙니다.

게시 전 로컬 실행: `DATA_DIR`을 동봉한 `data` 폴더로 지정. 게시 후 Colab에서는 키 없이 공개 CSV를 읽습니다. 선택 부록은 기본 실행하지 않습니다.

## Step 1. 한글 폰트 준비
Colab인 경우에만 나눔고딕을 설치합니다. 로컬에서는 `KOREAN_FONT`에 한글 TTF 경로를 지정할 수 있습니다.

In [ ]:
import os, sys, subprocess
if 'google.colab' in sys.modules:
    subprocess.run(['apt-get','-qq','update'],check=True)
    subprocess.run(['apt-get','-qq','install','-y','fonts-nanum'],check=True)
DATA_DIR=os.environ.get('DATA_DIR','https://raw.githubusercontent.com/team-datapopcorn/datapopcorn-assets/main/retail-outlook-gap/data')
OUT_DIR=os.environ.get('CHART_DIR','charts')
print('Data:',DATA_DIR)

## Step 2. 원자료 검증과 계산 함수
2개 CSV는 원본 전사값입니다. 결측·키 중복·값 범위와 전체 지수 교차일치를 확인합니다. 변화는 포인트, 기준선 거리는 지수−100으로 계산합니다.

In [ ]:
from pathlib import Path
import os, json, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np

SOURCE='대한상공회의소, 2026년 4분기 RBSI 조사 | 조회: 2026-10-22 | 계절조정 전 전망지수'
ORANGE='#E8590C'; BLUE='#1C7ED6'; GRAY='#868E96'; INK='#171717'

def setup_font():
    opts=[os.environ.get('KOREAN_FONT',''),'/usr/share/fonts/truetype/nanum/NanumGothic.ttf']
    for p in opts:
        if p and Path(p).exists():
            font_manager.fontManager.addfont(p)
            for weight in (600,800):
                bold=Path(p).with_name(f'Pretendard-{weight}.ttf')
                if bold.exists():font_manager.fontManager.addfont(str(bold))
            plt.rcParams['font.family']=font_manager.FontProperties(fname=p).get_name();break
    else: raise RuntimeError('한글 폰트를 찾지 못했습니다. Colab의 폰트 설치 셀을 실행하세요.')
    plt.rcParams.update({'axes.unicode_minus':False,'font.size':13,'axes.spines.top':False,'axes.spines.right':False,'axes.titleweight':'bold','axes.titlesize':20,'figure.facecolor':'white','axes.axisbelow':True})

def analyze(data_dir):
    sectors=pd.read_csv(str(data_dir).rstrip('/')+'/rbsi_sectors.csv',encoding='utf-8-sig')
    quarters=pd.read_csv(str(data_dir).rstrip('/')+'/rbsi_quarterly.csv',encoding='utf-8-sig')
    assert list(sectors.columns)==['sector','q3_2026','q4_2026']
    assert len(sectors)==6 and not sectors['sector'].duplicated().any() and not sectors.isna().any().any()
    assert list(quarters.columns)==['quarter','rbsi']
    assert len(quarters)==10 and not quarters['quarter'].duplicated().any() and not quarters.isna().any().any()
    for c in ['q3_2026','q4_2026']:sectors[c]=pd.to_numeric(sectors[c]); assert sectors[c].between(0,200).all()
    quarters['rbsi']=pd.to_numeric(quarters['rbsi']);assert quarters['rbsi'].between(0,200).all()
    sectors['change_points']=sectors['q4_2026']-sectors['q3_2026']
    sectors['distance_from_100']=sectors['q4_2026']-100
    detail=sectors[sectors['sector']!='전체'].copy()
    assert quarters.iloc[-1]['rbsi']==int(sectors.loc[sectors.sector=='전체','q4_2026'].iloc[0])
    summary={'overall_q3':int(sectors.iloc[0].q3_2026),'overall_q4':int(sectors.iloc[0].q4_2026),'overall_change_points':int(sectors.iloc[0].change_points),'sectors_increased':detail.loc[detail.change_points>0,'sector'].tolist(),'sectors_above_100':detail.loc[detail.q4_2026>100,'sector'].tolist(),'up_but_below_100':detail.loc[(detail.change_points>0)&(detail.q4_2026<100),'sector'].tolist(),'unweighted_sector_mean_q3':round(float(detail.q3_2026.mean()),1),'unweighted_sector_mean_q4':round(float(detail.q4_2026.mean()),1),'quarterly_observations':len(quarters),'sector_rows_including_overall':len(sectors)}
    return sectors,quarters,summary

def charts(sectors,quarters,outdir):
    setup_font();outdir=Path(outdir);outdir.mkdir(parents=True,exist_ok=True)
    detail=sectors[sectors.sector!='전체']; output=[]
    def finish(fig,name):
        fig.text(.08,.035,SOURCE,fontsize=10,color=GRAY)
        fig.subplots_adjust(left=.11,right=.94,top=.84,bottom=.20)
        p=outdir/(name+'.png');fig.savefig(p,dpi=160);output.append(p);plt.close(fig)
    fig,ax=plt.subplots(figsize=(12,6.8));x=np.arange(len(quarters));ax.plot(x,quarters.rbsi,color=BLUE,marker='o',lw=2.8)
    ax.axhline(100,color=GRAY,ls='--',lw=1.3);ax.text(0,102,'기준선 100',color=GRAY,fontsize=11)
    ax.scatter([x[-1]],[quarters.rbsi.iloc[-1]],color=ORANGE,s=65,zorder=5)
    for i,v in enumerate(quarters.rbsi):ax.text(i,v+3,str(v),ha='center',fontweight='bold',color=ORANGE if i==9 else INK)
    ax.set_xticks(x,quarters.quarter.str.replace('Q','년\n',regex=False)+'분기',fontsize=10);ax.set_ylim(0,130);ax.set_ylabel('RBSI (기준선 100)');ax.set_title('전체 전망은 92 → 88, 기준선 100 아래입니다',loc='left',pad=25);ax.grid(axis='y',alpha=.15);finish(fig,'01-quarterly')
    fig,ax=plt.subplots(figsize=(12,6.8));x=np.arange(len(detail));w=.34
    ax.bar(x-w/2,detail.q3_2026,w,color='#BCC2C8',label='2026년 3분기 전망');ax.bar(x+w/2,detail.q4_2026,w,color=ORANGE,label='2026년 4분기 전망');ax.axhline(100,color=INK,ls='--',lw=1)
    for i,(_,r) in enumerate(detail.iterrows()):
        ax.text(i-w/2,r.q3_2026+3,str(r.q3_2026),ha='center',fontsize=12);ax.text(i+w/2,r.q4_2026+3,str(r.q4_2026),ha='center',fontsize=12,fontweight='bold')
    ax.set_xticks(x,detail.sector);ax.set_ylim(0,175);ax.set_ylabel('RBSI');ax.set_title('백화점·대형마트는 100 위, 나머지 3개 업태는 아래',loc='left',pad=25);ax.legend(frameon=False,ncol=2,fontsize=11,loc='upper right');finish(fig,'02-sectors')
    fig,ax=plt.subplots(figsize=(12,6.8));sort=detail.sort_values('change_points');y=np.arange(len(sort));vals=sort.change_points
    ax.barh(y,vals,color=[BLUE if v>0 else ORANGE for v in vals]);ax.axvline(0,color=INK,lw=1)
    for i,v in enumerate(vals):ax.text(v+(1.2 if v>=0 else -1.2),i,f'{v:+d}p',ha='left' if v>=0 else 'right',va='center',fontweight='bold')
    ax.set_yticks(y,sort.sector);ax.set_xlim(-60,20);ax.set_xlabel('2026년 4분기 전망 − 3분기 전망 (포인트, 매출 증감률 아님)');ax.set_title('전망지수가 오른 업태는 온라인쇼핑 하나뿐입니다',loc='left',pad=25);ax.grid(axis='x',alpha=.15);finish(fig,'03-changes')
    fig,ax=plt.subplots(figsize=(12,7));ax.axvline(100,color=GRAY,ls='--');ax.axhline(0,color=GRAY,ls='--');ax.scatter(detail.q4_2026,detail.change_points,s=110,c=[BLUE if v>0 else ORANGE for v in detail.change_points],zorder=5)
    for _,r in detail.iterrows():
        dx,dy={'백화점':(8,-14),'대형마트':(6,8),'온라인쇼핑':(6,8),'편의점':(6,8),'슈퍼마켓':(6,-18)}[r.sector]
        ax.annotate(f'{r.sector} ({r.q4_2026}, {r.change_points:+d}p)',(r.q4_2026,r.change_points),xytext=(dx,dy),textcoords='offset points',fontsize=11)
    ax.set_xlim(60,150);ax.set_ylim(-60,25);ax.set_xlabel('2026년 4분기 전망 수준 (기준선 100)');ax.set_ylabel('3분기 전망 대비 변화 (포인트)');ax.set_title('온라인은 개선돼도 비관 우세, 백화점은 하락해도 낙관 우세',loc='left',pad=25,fontsize=18);ax.grid(alpha=.1);finish(fig,'04-two-axes')
    return output



## Step 3. 원자료를 읽고 두 축을 계산
500곳은 조사대상 수입니다. 공개된 원문에 실제 응답 총수와 업태별 표본 수는 없어 오차범위를 추정하지 않습니다. 전체 지수는 5개 업태 단순평균으로 대체하지 않습니다.

In [ ]:
sectors,quarters,summary=analyze(DATA_DIR)
display(sectors)
print(summary)
assert summary['up_but_below_100']==['온라인쇼핑']
assert summary['unweighted_sector_mean_q4'] != summary['overall_q4']

## Step 4. 네 가지 차트 재현
① 10개 분기 전체 추이 ② 5개 업태의 두 분기 비교 ③ 전분기 대비 포인트 변화 ④ 수준과 변화의 두 축. 원계열이라 계절 효과가 남아 있습니다.

In [ ]:
from IPython.display import display,Image
chart_paths=charts(sectors,quarters,OUT_DIR)
assert len(chart_paths)==4
for p in chart_paths: display(Image(filename=str(p)))
print(f'완료: 차트 {len(chart_paths)}개')

## Step 5. 해석의 한계
- 조사기간 2026-08-31~09-18, 인터넷 보도 허용 10월21일12시, 공식 게시10월22일. 발표일의 실시간 응답이 아닙니다.
- 서울 및 6대 광역시의 유통업체 패널 조사, 전국 모든 점포의 센서스가 아닙니다.
- RBSI=(호전예상 기업수−악화예상 기업수)/응답기업수×100+100. 88은 매출12%하락이 아닙니다.
- 계절조정 전 전망이며 인과·매출 예측 정확도는 검증하지 않습니다.
- 분기 지수끼리 비교하는 증감은 각기 다른 시점의 전망 수준 비교입니다.
- 전체88과 업태별 단순평균95의 차이는 확인되지만, 업태별 표본·가중치 미공개로 재구성하지 않습니다.

## 부록. 공식 원문 다시 받기(선택, 기본 OFF)
키가 필요 없는 공식 PDF입니다. 원문을 개인 작업 폴더에 보존하고 본문·그림 수치를 대조합니다. 기사 본문/공식 PDF는 이 패키지에 재배포하지 않았습니다. 네이버 수요 조회는 화제성 보조 점검이었으며 본 분석과 차트의 입력이 아닙니다.

In [ ]:
RUN_COLLECTION=False
if RUN_COLLECTION:
    import requests,hashlib,datetime,json
    from pathlib import Path
    url='http://www.koreannet.or.kr/madang/fileDownload.do?savedFileName=4df6dca3-4293-43e5-8c94-e8d576bd7c23.pdf&filePath=/storage/koreannet3.0/2026'
    r=requests.get(url,timeout=45);r.raise_for_status()
    assert r.content.startswith(b'%PDF'), 'PDF 대신 오류/로그인 페이지가 반환됐습니다'
    raw=Path('raw');raw.mkdir(exist_ok=True)
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    path=raw/f'kcci_rbsi_{stamp}.pdf';path.write_bytes(r.content)
    print({'path':str(path),'sha256':hashlib.sha256(r.content).hexdigest()})